# 1. Agentes con Memoria Conversacional en LangChain

## Objetivos de Aprendizaje
- Comprender por qué la memoria es crucial para crear agentes conversacionales efectivos.
- Aprender a gestionar el historial de una conversación (`chat_history`) con el `AgentExecutor`.
- Implementar un agente que recuerde interacciones pasadas para responder preguntas de seguimiento.
- Entender cómo LangChain pasa el contexto de la conversación al LLM.

## ¿Qué es la Memoria y Por Qué es Importante?

Por defecto, los LLMs y los agentes que hemos construido hasta ahora **no tienen estado (stateless)**. Cada vez que los invocamos, procesan la solicitud como si fuera la primera vez que interactúan con nosotros. No tienen recuerdo de preguntas o respuestas anteriores.

Esto es una gran limitación para crear asistentes o chatbots útiles. Un usuario espera poder hacer preguntas de seguimiento, referirse a información mencionada previamente y tener una conversación fluida. 

La **memoria** es el mecanismo que permite a un agente recordar interacciones pasadas. LangChain facilita enormemente la gestión de esta memoria. La forma más común de memoria es el **historial de chat (chat history)**, donde simplemente guardamos la lista de todos los mensajes de la conversación.

En este notebook, veremos cómo añadir esta capacidad a nuestro agente de LangChain.

### 1. Instalación y Configuración

In [1]:
!pip install -qU langchain-groq groq langgraph langchain langchain-classic wikipedia python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import wikipedia
from langchain_groq import ChatGroq

# Carga de credenciales: funciona igual en Google Colab y en local (.env)
try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

# Agente de LangChain con herramientas nativas (create_openai_tools_agent).
# Por esta vía el modelo grande es el fiable: medido sobre 6 consultas, llama-3.3-70b
# acertó 6/6 el formato de la llamada a la función y llama-3.1-8b solo 4/6.
# (Ojo: con el SDK crudo de Groq la relación se invierte; ver 2-agent-function-calling.)
MODELO = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM (ChatGroq lee GROQ_API_KEY del entorno)
try:
    llm = ChatGroq(
        model=MODELO,
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado.


### 2. Herramientas y Agente (Sin Cambios)

La definición de las herramientas y la creación del agente son exactamente las mismas que en el notebook anterior. La magia de la memoria no está en la definición del agente, sino en **cómo lo ejecutamos**.

In [3]:
from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langchain_classic import hub

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
prompt = hub.pull("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

✅ Agente y herramientas listos.


### 3. Gestionando la Memoria Conversacional

Para que el agente recuerde, necesitamos hacer dos cosas:

1.  **Mantener un historial**: Crearemos una lista llamada `chat_history` para almacenar los mensajes.
2.  **Pasar el historial en cada llamada**: El `AgentExecutor` acepta un parámetro `chat_history`. LangChain se encarga de formatear esta lista y añadirla al prompt que se envía al LLM.

El formato del historial es una lista de objetos `BaseMessage` de LangChain. Los más comunes son `HumanMessage` (para el usuario) y `AIMessage` (para la respuesta del agente).

In [4]:
from langchain_core.messages import HumanMessage, AIMessage

# Iniciamos el historial de chat como una lista vacía
chat_history = []

#### Primera Interacción: Sin Historial

In [5]:
query1 = "Háblame del planeta Saturno"

response1 = agent_executor.invoke({
    "input": query1,
    "chat_history": chat_history
})

print(f"Respuesta 1: {response1['output']}")



> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'Saturno'}`




/Users/giocrisraigodoy/Documents/DUOC/2026-1/INGENIERIA DE SOLUCIONES CON INTELIGENCIA ARTIFICIAL/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/.venv/lib/python3.13/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /Users/giocrisraigodoy/Documents/DUOC/2026-1/INGENIERIA DE SOLUCIONES CON INTELIGENCIA ARTIFICIAL/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/.venv/lib/python3.13/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


Ocurrió un error: "Saturno" may refer to: 
Saturno (mitología)
Saturno (planeta)
Saturno (Buenos Aires)
Saturno (Rubens)
Saturno (cohete)
Saturno I
Saturno IB
Saturno V
Sombrero saturno
Sergio Saturno
Saturno (álbum)
Wikcionario


Invoking: `get_wikipedia_summary` with `{'query': 'Saturno planeta'}`




Saturno es el sexto planeta del sistema solar contando desde el Sol, el segundo en tamaño y masa después de Júpiter y el único con un sistema de anillos visible desde la Tierra. Su nombre proviene del dios romano Saturno.

Saturno es un planeta gaseoso, compuesto principalmente por hidrógeno y helio, con vientos que pueden alcanzar velocidades de hasta 1.800 kilómetros por hora, lo que lo convierte en uno de los planetas más ventosos del sistema solar. Su sistema de anillos, compuesto por hielo y roca, es uno de los más impresionantes del sistema solar, con un diámetro de aproximadamente 282.000 kilómetros. Saturno tiene al menos 62 lunas, siendo la más grande Titán, que es la segunda luna más grande del sistema solar después de Ganímedes, que orbita Júpiter. La exploración de Saturno ha sido llevada a cabo por varias misiones espaciales, incluyendo la sonda Cassini, que orbitó el planeta desde 2004 hasta 2017 y proporcionó una gran cantidad de información sobre su atmósfera, su sistema de anillos y sus lunas.

> Finished chain.
Respuesta 1: Saturno es un planeta gaseoso, compuesto principalmente por hidrógeno y helio, con vientos que pueden alcanzar velocidades de hasta 1.800 kilómetros por hora, lo que

Ahora, actualizamos manualmente nuestro historial con la pregunta del usuario y la respuesta del agente.

In [6]:
chat_history.append(HumanMessage(content=query1))
chat_history.append(AIMessage(content=response1["output"]))

print("Historial actualizado.")

Historial actualizado.


#### Segunda Interacción: Con Historial

Ahora hacemos una pregunta de seguimiento. Fíjate que no mencionamos "Saturno", simplemente preguntamos "¿de qué están hechos sus anillos?"

In [7]:
query2 = "¿Y de qué están hechos sus anillos?"

response2 = agent_executor.invoke({
    "input": query2,
    "chat_history": chat_history
})

print(f"Respuesta 2: {response2['output']}")



> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'anillos de Saturno composicion'}`




Los anillos de Saturno son un sistema de 7 anillos planetarios que rodean a ese planeta y fueron observados por primera vez en julio de 1610 por Galileo Galilei. En parte porque las imágenes que daba el recién inventado telescopio eran de mala calidad para aquel entonces, y en parte porque hacía solo unos meses que había descubierto los cuatro mayores satélites de Júpiter, pensó inicialmente que las estructuras borrosas, parecidas a orejas, que había visto eran dos satélites próximos a Saturno.


Invoking: `get_wikipedia_summary` with `{'query': 'composicion de los anillos de Saturno'}`




Los anillos de Saturno son un sistema de 7 anillos planetarios que rodean a ese planeta y fueron observados por primera vez en julio de 1610 por Galileo Galilei. En parte porque las imágenes que daba el recién inventado telescopio eran de mala calidad para aquel entonces, y en parte porque hacía solo unos meses que había descubierto los cuatro mayores satélites de Júpiter, pensó inicialmente que las estructuras borrosas, parecidas a orejas, que había visto eran dos satélites próximos a Saturno.


Invoking: `get_wikipedia_summary` with `{'query': 'anillos de Saturno'}`




Los anillos de Saturno son un sistema de 7 anillos planetarios que rodean a ese planeta y fueron observados por primera vez en julio de 1610 por Galileo Galilei. En parte porque las imágenes que daba el recién inventado telescopio eran de mala calidad para aquel entonces, y en parte porque hacía solo unos meses que había descubierto los cuatro mayores satélites de Júpiter, pensó inicialmente que las estructuras borrosas, parecidas a orejas, que había visto eran dos satélites próximos a Saturno.

Los anillos de Saturno están compuestos principalmente por hielo y roca, con partículas que varían en tamaño desde pequeños granos de polvo hasta rocas de varios metros de diámetro. La mayoría de las partículas que componen los anillos son de hielo, pero también hay una pequeña cantidad de roca y otros materiales. Los anillos son muy delgados, con un grosor de apenas 30 pies (10 metros) en algunos lugares, y están compuestos por miles de anillos individuales que se extienden a lo largo de cientos de miles de kilómetros. La formación de los anillos de Saturno es aún un tema de debate entre los científicos, pero se cree que se formaron a partir de la desintegración de lunas y otros objetos que orbitaban el planeta en el pasado.

> Finished chain.
Respuesta 2: Los anillos de Saturno están compuestos principalmente por hielo y roca, con partículas que varían en tamaño desde pequeños granos de polvo hasta rocas de varios metros de diámetro. La mayoría de las partículas que componen los anil

¡Funcionó! El agente entendió que "sus anillos" se refería a los anillos de Saturno, porque la conversación anterior estaba en su contexto. El `verbose=True` nos muestra que el agente decidió buscar en Wikipedia "anillos de Saturno", combinando la nueva pregunta con el historial.

## Conclusiones

Añadir memoria a un agente de LangChain es sorprendentemente sencillo, pero increíblemente poderoso. Simplemente manteniendo una lista del historial de chat y pasándola en cada invocación, transformamos un agente de una sola respuesta en un verdadero **asistente conversacional**.

La clave es que los componentes de LangChain (`AgentExecutor`, los prompts de `hub`) ya están diseñados para buscar y utilizar la variable `chat_history` si se proporciona.

**Limitaciones:**
- **Gestión Manual**: En este ejemplo, actualizamos la lista `chat_history` manualmente. Para una aplicación real, querríamos encapsular esto en una clase o función.
- **Tamaño del Contexto**: Enviar el historial completo en cada llamada puede volverse costoso y exceder el límite de tokens del modelo en conversaciones muy largas.

En los próximos notebooks, exploraremos los **sistemas de memoria** que LangChain ofrece para gestionar estas limitaciones, como la memoria de búfer (`BufferMemory`) que automatiza la gestión del historial y la memoria de resumen (`SummaryMemory`) que condensa conversaciones largas para ahorrar tokens.